In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

dataframe = pd.read_csv(
    "../data/sentences.csv.zip",
    index_col=False,
    compression="zip",
)

X_train, X_test, y_train, y_test = train_test_split(
    dataframe.phrase,
    dataframe.target,
    test_size=0.2,
    random_state=42,
    stratify=dataframe.target,
)
dataframe.head()

,phrase,target
0,"According to Gran , the company has no plans t...",neutral
1,"For the last quarter of 2010 , Componenta 's n...",positive
2,"In the third quarter of 2010 , net sales incre...",positive
3,Operating profit rose to EUR 13.1 mn from EUR ...,positive
4,"Operating profit totalled EUR 21.1 mn , up fro...",positive


In [2]:
# Pipeline + hyperparameter search
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipeline = Pipeline(
    [
        ("vectorizer", CountVectorizer(lowercase=True)),
        ("tfidf", TfidfTransformer()),
        ("clf", LogisticRegression(max_iter=1000)),
    ]
)

param_grid = {
    "vectorizer__ngram_range": [(1, 1), (1, 2)],
    "vectorizer__min_df": [1, 2],
    "tfidf__use_idf": [True, False],
    "clf__C": [1, 10, 100],
}

estimator = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)
estimator.fit(X_train, y_train)

/home/samuel/Desktop/Programming/Analitica/std-samuel-hernandez-duque/.venv/lib/python3.11/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('vectorizer', CountVectorizer()),
                                       ('tfidf', TfidfTransformer()),
                                       ('clf',
                                        LogisticRegression(max_iter=1000))]),
             n_jobs=-1,
             param_grid={'clf__C': [1, 10, 100],
                         'tfidf__use_idf': [True, False],
                         'vectorizer__min_df': [1, 2],
                         'vectorizer__ngram_range': [(1, 1), (1, 2)]},
             scoring='accuracy')

In [3]:
estimator.best_params_, estimator.best_score_

({'clf__C': 100,
  'tfidf__use_idf': True,
  'vectorizer__min_df': 2,
  'vectorizer__ngram_range': (1, 2)},
 0.8763115839459384)

In [4]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_test = estimator.predict(X_test)
print(classification_report(y_test, y_pred_test))
accuracy_score(y_test, y_pred_test)

              precision    recall  f1-score   support

    negative       0.89      0.80      0.84        61
     neutral       0.95      0.99      0.97       278
    positive       0.87      0.84      0.86       114

    accuracy                           0.92       453
   macro avg       0.91      0.88      0.89       453
weighted avg       0.92      0.92      0.92       453



0.9249448123620309

In [5]:
import os
import pickle

os.makedirs("../submission", exist_ok=True)

with open("../submission/estimator.pkl", "wb") as file:
    pickle.dump(estimator, file)